# RecallZero — Experiment 02: Monitored LoRA unlearning

The first 40-step pilot barely changed actual model responses. This second experiment **does not promise successful forgetting**. It increases optimization intensity, saves checkpoints at multiple stages, and records both loss changes and example generations.

**Inputs:** Attach the same private Kaggle dataset containing `forget_train.json`, `retain_train.json`, and `retain_eval.json` from your Windows `data/processed/` folder. Enable a Kaggle GPU and internet access.

**Outputs:** `recallzero_v2_candidates.zip` and `recallzero_v2_metrics.json` in `/kaggle/working`. The original TOFU model and your previous pilot adapter are never overwritten.

**Experimental limitations:** Forget CE is evaluated on trained questions. A held-out paraphrase/TOFU perturbed evaluation is still required; we do not claim complete erasure from loss or changed strings alone.


In [ ]:
# Kaggle may include an old optional torchao that breaks PEFT adapter injection.
%pip -q uninstall -y torchao
%pip -q install "transformers==4.57.6" "peft>=0.17,<1" "accelerate>=1.10,<2" safetensors
# If this notebook previously errored during PEFT injection, restart the Kaggle session, then Run All.


In [ ]:
import json, os, random, zipfile, math
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), "Enable GPU in Kaggle Notebook settings."
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

ROOT = Path("/kaggle/input")
def get_records(name):
    matches = list(ROOT.rglob(name))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one {name} under /kaggle/input, found {len(matches)}")
    rows = json.loads(matches[0].read_text(encoding="utf-8"))
    assert isinstance(rows, list) and all("question" in x and "answer" in x for x in rows)
    print(f"{name}: {len(rows)} rows")
    return rows

forget_rows = get_records("forget_train.json")
retain_rows = get_records("retain_train.json")
retain_eval_rows = get_records("retain_eval.json")
assert len(forget_rows) == len(retain_rows) == len(retain_eval_rows) == 40


In [ ]:
MODEL_ID = "open-unlearning/tofu_Llama-3.2-1B-Instruct_full"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
base_model.config.use_cache = False
base_model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
base_model.to("cuda:0")

# Slightly broader adapter than experiment 01 (q/v only).
config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, config)
model.print_trainable_parameters()


In [ ]:
MAX_LENGTH = 384

def encode_qa(row):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": row["question"]},
    ]
    prefix = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prefix_ids = tokenizer(prefix, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(row["answer"], add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
    ids = (prefix_ids + answer_ids)[:MAX_LENGTH]
    labels = ([-100] * len(prefix_ids) + answer_ids)[:MAX_LENGTH]
    assert len(ids) == len(labels) and any(x != -100 for x in labels)
    return dict(
        input_ids=torch.tensor([ids], dtype=torch.long, device="cuda:0"),
        attention_mask=torch.ones((1,len(ids)), dtype=torch.long, device="cuda:0"),
        labels=torch.tensor([labels], dtype=torch.long, device="cuda:0"),
    )

fb = [encode_qa(x) for x in forget_rows]
rb = [encode_qa(x) for x in retain_rows]
re = [encode_qa(x) for x in retain_eval_rows]

def ce(batch): return model(**batch).loss

@torch.no_grad()
def mean_ce(batches):
    model.eval()
    values = [float(ce(x).item()) for x in batches]
    return sum(values)/len(values)

@torch.inference_mode()
def answer(question):
    model.eval()
    messages = [
        {"role":"system","content":"You are a helpful assistant."},
        {"role":"user","content":question},
    ]
    tokens = tokenizer.apply_chat_template(
        messages, return_tensors="pt", return_dict=True, add_generation_prompt=True
    ).to("cuda:0")
    result = model.generate(
        **tokens,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )
    return tokenizer.decode(result[0, tokens["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Same fixed questions at every checkpoint; not claimed as a held-out test.
SHOW_FORGET = [0, 3, 5, 6]
SHOW_RETAIN = [0, 10, 20, 30]

def snapshot(step):
    fc = mean_ce(fb)     # all 40 forget training questions
    rc = mean_ce(re)     # all 40 distinct retained evaluation questions
    examples = {"forget":[], "retain":[]}
    for i in SHOW_FORGET:
        examples["forget"].append({"index":i,"question":forget_rows[i]["question"],
                                   "expected":forget_rows[i]["answer"],"generated":answer(forget_rows[i]["question"])})
    for i in SHOW_RETAIN:
        examples["retain"].append({"index":i,"question":retain_eval_rows[i]["question"],
                                   "expected":retain_eval_rows[i]["answer"],"generated":answer(retain_eval_rows[i]["question"])})
    print(f"Step {step:03d} | Forget CE: {fc:.4f} | Retain CE: {rc:.4f}")
    return {"step":step,"forget_train_ce":fc,"retain_eval_ce":rc,"samples":examples}

snapshots = [snapshot(0)]


In [ ]:
# Experiment 02 settings: intentionally stronger than the unsuccessful 40-step pilot.
STEPS = 160
LEARNING_RATE = 1.5e-4
RETAIN_WEIGHT = 0.75
CHECKPOINT_EVERY = 40

OUT = Path("/kaggle/working")
SAVE_ROOT = OUT / "recallzero_v2_checkpoints"
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=0.0,
)
training_log = []
order_forget=list(range(len(fb)))
order_retain=list(range(len(rb)))

for step in range(1, STEPS+1):
    model.train()
    if (step - 1) % len(fb) == 0:
        random.shuffle(order_forget)
        random.shuffle(order_retain)
    fidx = order_forget[(step-1) % len(fb)]
    ridx = order_retain[(step-1) % len(rb)]
    optimizer.zero_grad(set_to_none=True)
    forget_ce = ce(fb[fidx])
    retain_ce = ce(rb[ridx])
    objective = -forget_ce + RETAIN_WEIGHT * retain_ce
    if not torch.isfinite(objective):
        raise FloatingPointError(f"Nonfinite loss at step {step}; stopping")
    objective.backward()
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], max_norm=1.0
    )
    optimizer.step()
    training_log.append({
        "step":step,
        "forget_train_batch_ce":float(forget_ce.detach()),
        "retain_train_batch_ce":float(retain_ce.detach()),
        "objective":float(objective.detach()),
    })

    if step % CHECKPOINT_EVERY == 0:
        snap = snapshot(step)
        snapshots.append(snap)
        folder = SAVE_ROOT / f"step_{step:03d}"
        folder.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(folder)
        tokenizer.save_pretrained(folder)
        # This is a warning threshold, not a guaranteed quality criterion.
        if snap["retain_eval_ce"] > snapshots[0]["retain_eval_ce"] * 1.30:
            print("WARNING: retain evaluation CE grew more than 30%. Stopping to limit damage.")
            break

print("Training stopped at step:", snapshots[-1]["step"])


In [ ]:
# Export *all* intermediate adapters, not just a final candidate.
metrics = {
    "model_id":MODEL_ID,
    "experiment":"02 - stronger gradient ascent + retention CE",
    "seed":SEED,
    "max_steps":STEPS,
    "completed_steps":snapshots[-1]["step"],
    "learning_rate":LEARNING_RATE,
    "retain_weight":RETAIN_WEIGHT,
    "checkpoints":snapshots,
    "training_log":training_log,
    "limitations":[
        "All forget CE measurements are on forget training questions.",
        "Retain CE is on 40 held-out retain questions; this is not a complete utility evaluation.",
        "Paraphrased-question, perturbed-answer, and external tests remain necessary.",
        "Changed wording or increased forget CE alone does not prove model knowledge has been erased.",
    ],
}
metrics_path = OUT / "recallzero_v2_metrics.json"
metrics_path.write_text(json.dumps(metrics,indent=2,ensure_ascii=False),encoding="utf-8")

zip_path = OUT / "recallzero_v2_candidates.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in SAVE_ROOT.rglob("*"):
        if p.is_file(): z.write(p,p.relative_to(OUT))
print("Saved:",metrics_path)
print("Saved:",zip_path)
print("Compare checkpoints before picking a final model. No unlearning success is assumed.")
